# Snowflake VARIANT Column Testing

This notebook demonstrates how to write to Snowflake tables with VARIANT columns using different ORM approaches.

## Common Challenges:
- VARIANT type requires proper JSON serialization
- Different ORMs handle VARIANT differently
- Type conversion issues with nested objects
- Null handling in VARIANT fields

## What We'll Cover:
1. Setup and connection
2. Table creation with VARIANT columns
3. Writing with SQLAlchemy
4. Writing with Snowflake Connector (raw)
5. Writing with pandas
6. Control plane bulk writes
7. Troubleshooting common errors

## 1. Setup and Imports

In [ ]:
# Install required packages (run once)
!pip install snowflake-connector-python snowflake-sqlalchemy sqlalchemy pandas

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime
from typing import Dict, Any, List

# Snowflake imports
import snowflake.connector
from snowflake.connector import DictCursor

# SQLAlchemy imports
from sqlalchemy import create_engine, Column, Integer, String, DateTime, Text
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
from sqlalchemy.types import TypeDecorator, VARCHAR

print("✅ Imports successful")

## 2. Configuration

Set your Snowflake credentials here or load from environment variables.

In [ ]:
# Snowflake connection parameters
SNOWFLAKE_CONFIG = {
    'account': os.getenv('SNOWFLAKE_ACCOUNT', 'your-account'),
    'user': os.getenv('SNOWFLAKE_USER', 'your-user'),
    'password': os.getenv('SNOWFLAKE_PASSWORD', 'your-password'),
    'database': os.getenv('SNOWFLAKE_DATABASE', 'TEST_DB'),
    'schema': os.getenv('SNOWFLAKE_SCHEMA', 'PUBLIC'),
    'warehouse': os.getenv('SNOWFLAKE_WAREHOUSE', 'COMPUTE_WH'),
    'role': os.getenv('SNOWFLAKE_ROLE', 'SYSADMIN')
}

# Test table name
TEST_TABLE = 'VARIANT_TEST_TABLE'

print(f"Config loaded: {SNOWFLAKE_CONFIG['account']}/{SNOWFLAKE_CONFIG['database']}/{SNOWFLAKE_CONFIG['schema']}")

## 3. Create Test Table with VARIANT Columns

In [ ]:
def create_test_table():
    """Create test table with VARIANT columns"""
    conn = snowflake.connector.connect(**SNOWFLAKE_CONFIG)
    cursor = conn.cursor()
    
    try:
        # Drop table if exists
        cursor.execute(f"DROP TABLE IF EXISTS {TEST_TABLE}")
        print(f"✅ Dropped existing table {TEST_TABLE}")
        
        # Create table with mixed column types
        create_sql = f"""
        CREATE TABLE {TEST_TABLE} (
            id INTEGER AUTOINCREMENT PRIMARY KEY,
            event_name VARCHAR(255) NOT NULL,
            event_timestamp TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
            
            -- VARIANT columns (the tricky ones!)
            event_data VARIANT,
            user_metadata VARIANT,
            raw_payload VARIANT,
            
            -- Regular columns for comparison
            user_id INTEGER,
            status VARCHAR(50)
        )
        """
        
        cursor.execute(create_sql)
        print(f"✅ Created table {TEST_TABLE}")
        
        # Show table structure
        cursor.execute(f"DESC TABLE {TEST_TABLE}")
        columns = cursor.fetchall()
        
        print("\n📋 Table Structure:")
        for col in columns:
            col_name = col[0]
            col_type = col[1]
            print(f"  {col_name}: {col_type}")
        
        return True
        
    except Exception as e:
        print(f"❌ Error creating table: {e}")
        return False
    finally:
        cursor.close()
        conn.close()

# Create the table
create_test_table()

## 4. Method 1: Raw Snowflake Connector (Most Reliable)

This is the most straightforward approach - use `json.dumps()` to serialize Python objects.

In [ ]:
def write_with_raw_connector():
    """Write using raw Snowflake connector - RECOMMENDED for VARIANT"""
    conn = snowflake.connector.connect(**SNOWFLAKE_CONFIG)
    cursor = conn.cursor()
    
    try:
        # Sample data with complex nested structures
        test_data = [
            {
                'event_name': 'user_login',
                'user_id': 123,
                'status': 'success',
                'event_data': {
                    'browser': 'Chrome',
                    'os': 'macOS',
                    'ip_address': '192.168.1.1',
                    'session_duration': 3600
                },
                'user_metadata': {
                    'name': 'John Doe',
                    'email': 'john@example.com',
                    'preferences': {
                        'theme': 'dark',
                        'notifications': True
                    }
                },
                'raw_payload': {
                    'nested': {
                        'deeply': {
                            'nested': {
                                'value': [1, 2, 3, 4, 5]
                            }
                        }
                    }
                }
            },
            {
                'event_name': 'page_view',
                'user_id': 456,
                'status': 'success',
                'event_data': {
                    'page': '/dashboard',
                    'referrer': 'https://example.com',
                    'time_on_page': 120
                },
                'user_metadata': {
                    'name': 'Jane Smith',
                    'tags': ['premium', 'active'],
                    'score': 95.5
                },
                'raw_payload': None  # Test NULL handling
            }
        ]
        
        # ✅ KEY POINT: Use PARSE_JSON() in SQL and json.dumps() in Python
        insert_sql = f"""
        INSERT INTO {TEST_TABLE} 
        (event_name, user_id, status, event_data, user_metadata, raw_payload)
        VALUES (%s, %s, %s, PARSE_JSON(%s), PARSE_JSON(%s), PARSE_JSON(%s))
        """
        
        # Prepare data with JSON serialization
        rows = []
        for record in test_data:
            rows.append((
                record['event_name'],
                record['user_id'],
                record['status'],
                json.dumps(record['event_data']) if record['event_data'] else None,
                json.dumps(record['user_metadata']) if record['user_metadata'] else None,
                json.dumps(record['raw_payload']) if record['raw_payload'] else None
            ))
        
        # Execute batch insert
        cursor.executemany(insert_sql, rows)
        conn.commit()
        
        print(f"✅ Inserted {len(rows)} rows using raw connector")
        
        # Verify the insert
        cursor.execute(f"""
            SELECT 
                event_name, 
                user_id,
                event_data,
                user_metadata
            FROM {TEST_TABLE}
            WHERE event_name IN ('user_login', 'page_view')
            ORDER BY id DESC
            LIMIT 2
        """)
        
        results = cursor.fetchall()
        print("\n📊 Verification Query Results:")
        for row in results:
            print(f"\nEvent: {row[0]}")
            print(f"User ID: {row[1]}")
            print(f"Event Data: {row[2]}")
            print(f"User Metadata: {row[3]}")
        
        return True
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return False
    finally:
        cursor.close()
        conn.close()

# Test the raw connector method
write_with_raw_connector()

## 5. Method 2: SQLAlchemy with Custom VARIANT Type

SQLAlchemy doesn't have native VARIANT support, so we need to create a custom type.

In [ ]:
# Define custom VARIANT type for SQLAlchemy
class JSONVariant(TypeDecorator):
    """Custom type for Snowflake VARIANT columns"""
    impl = VARCHAR
    cache_ok = True
    
    def process_bind_param(self, value, dialect):
        """Convert Python object to JSON string for database"""
        if value is not None:
            return json.dumps(value)
        return None
    
    def process_result_value(self, value, dialect):
        """Convert JSON string from database to Python object"""
        if value is not None:
            return json.loads(value)
        return None

# Define ORM model
Base = declarative_base()

class VariantTestTable(Base):
    __tablename__ = TEST_TABLE.lower()
    
    id = Column(Integer, primary_key=True)
    event_name = Column(String(255), nullable=False)
    event_timestamp = Column(DateTime)
    
    # VARIANT columns using custom type
    event_data = Column(JSONVariant)
    user_metadata = Column(JSONVariant)
    raw_payload = Column(JSONVariant)
    
    user_id = Column(Integer)
    status = Column(String(50))
    
    def __repr__(self):
        return f"<Event(name={self.event_name}, user_id={self.user_id})>"

print("✅ SQLAlchemy model defined")

In [ ]:
def write_with_sqlalchemy():
    """Write using SQLAlchemy ORM"""
    # Create SQLAlchemy engine
    connection_string = (
        f"snowflake://{SNOWFLAKE_CONFIG['user']}:{SNOWFLAKE_CONFIG['password']}"
        f"@{SNOWFLAKE_CONFIG['account']}/{SNOWFLAKE_CONFIG['database']}"
        f"/{SNOWFLAKE_CONFIG['schema']}"
        f"?warehouse={SNOWFLAKE_CONFIG['warehouse']}"
        f"&role={SNOWFLAKE_CONFIG['role']}"
    )
    
    engine = create_engine(connection_string)
    Session = sessionmaker(bind=engine)
    session = Session()
    
    try:
        # Create test records
        records = [
            VariantTestTable(
                event_name='api_call',
                user_id=789,
                status='success',
                event_data={
                    'endpoint': '/api/v1/users',
                    'method': 'POST',
                    'response_time': 150,
                    'status_code': 200
                },
                user_metadata={
                    'api_key': 'key_123',
                    'client_version': '2.1.0',
                    'features': ['feature_a', 'feature_b']
                },
                raw_payload={
                    'request': {
                        'headers': {'Content-Type': 'application/json'},
                        'body': {'name': 'Test User'}
                    }
                }
            ),
            VariantTestTable(
                event_name='error_event',
                user_id=999,
                status='failed',
                event_data={
                    'error_code': 'E500',
                    'error_message': 'Internal server error',
                    'stack_trace': ['line1', 'line2', 'line3']
                },
                user_metadata={
                    'user_agent': 'Mozilla/5.0',
                    'ip': '10.0.0.1'
                },
                raw_payload=None  # Test NULL
            )
        ]
        
        # Add to session and commit
        session.add_all(records)
        session.commit()
        
        print(f"✅ Inserted {len(records)} rows using SQLAlchemy")
        
        # Query back to verify
        results = session.query(VariantTestTable).filter(
            VariantTestTable.event_name.in_(['api_call', 'error_event'])
        ).all()
        
        print("\n📊 Verification Query Results:")
        for record in results:
            print(f"\nEvent: {record.event_name}")
            print(f"User ID: {record.user_id}")
            print(f"Event Data: {record.event_data}")
            print(f"User Metadata: {record.user_metadata}")
        
        return True
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        session.rollback()
        return False
    finally:
        session.close()

# Test SQLAlchemy method
write_with_sqlalchemy()

## 6. Method 3: Pandas with Snowflake

Pandas can write to Snowflake, but requires careful handling of VARIANT columns.

In [ ]:
def write_with_pandas():
    """Write using pandas to_sql()"""
    from sqlalchemy import create_engine
    
    # Create engine
    connection_string = (
        f"snowflake://{SNOWFLAKE_CONFIG['user']}:{SNOWFLAKE_CONFIG['password']}"
        f"@{SNOWFLAKE_CONFIG['account']}/{SNOWFLAKE_CONFIG['database']}"
        f"/{SNOWFLAKE_CONFIG['schema']}"
        f"?warehouse={SNOWFLAKE_CONFIG['warehouse']}"
    )
    
    engine = create_engine(connection_string)
    
    try:
        # Create DataFrame
        data = {
            'event_name': ['file_upload', 'file_download'],
            'user_id': [111, 222],
            'status': ['success', 'success'],
            'event_data': [
                json.dumps({
                    'file_name': 'document.pdf',
                    'file_size': 1024000,
                    'content_type': 'application/pdf'
                }),
                json.dumps({
                    'file_name': 'image.png',
                    'file_size': 512000,
                    'content_type': 'image/png'
                })
            ],
            'user_metadata': [
                json.dumps({'folder': '/documents', 'shared': True}),
                json.dumps({'folder': '/images', 'shared': False})
            ],
            'raw_payload': [
                json.dumps({'upload_source': 'web'}),
                None
            ]
        }
        
        df = pd.DataFrame(data)
        
        print("📊 DataFrame to insert:")
        print(df)
        
        # Write to Snowflake
        # Important: Use if_exists='append' to add to existing table
        df.to_sql(
            TEST_TABLE.lower(),
            engine,
            if_exists='append',
            index=False,
            method='multi',
            chunksize=1000
        )
        
        print(f"\n✅ Inserted {len(df)} rows using pandas")
        
        # Verify
        query = f"""
        SELECT event_name, user_id, event_data, user_metadata
        FROM {TEST_TABLE}
        WHERE event_name IN ('file_upload', 'file_download')
        ORDER BY id DESC
        LIMIT 2
        """
        
        result_df = pd.read_sql(query, engine)
        print("\n📊 Verification Query Results:")
        print(result_df)
        
        return True
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return False

# Test pandas method
write_with_pandas()

## 7. Common Errors and Solutions

Let's test and document common errors you might encounter.

In [ ]:
def test_common_errors():
    """Demonstrate common errors and their solutions"""
    conn = snowflake.connector.connect(**SNOWFLAKE_CONFIG)
    cursor = conn.cursor()
    
    print("🔍 Testing Common Errors:\n")
    
    # Error 1: Forgetting to serialize to JSON
    print("❌ ERROR 1: Inserting dict directly (without json.dumps)")
    try:
        bad_data = {'key': 'value'}
        cursor.execute(f"""
            INSERT INTO {TEST_TABLE} (event_name, user_id, event_data)
            VALUES ('test', 1, PARSE_JSON(%s))
        """, (bad_data,))  # ❌ This will fail!
    except Exception as e:
        print(f"   Error: {type(e).__name__}: {str(e)[:100]}")
        print("   ✅ Solution: Use json.dumps(bad_data)\n")
    
    # Error 2: Not using PARSE_JSON
    print("❌ ERROR 2: Inserting JSON string without PARSE_JSON()")
    try:
        cursor.execute(f"""
            INSERT INTO {TEST_TABLE} (event_name, user_id, event_data)
            VALUES ('test', 1, %s)
        """, (json.dumps({'key': 'value'}),))  # ❌ Missing PARSE_JSON!
        conn.commit()
        # This might succeed but store as STRING instead of VARIANT
        print("   ⚠️  Warning: This stores as STRING, not VARIANT")
        print("   ✅ Solution: Use PARSE_JSON(%s) in SQL\n")
    except Exception as e:
        print(f"   Error: {e}\n")
    
    # Error 3: Invalid JSON
    print("❌ ERROR 3: Invalid JSON string")
    try:
        cursor.execute(f"""
            INSERT INTO {TEST_TABLE} (event_name, user_id, event_data)
            VALUES ('test', 1, PARSE_JSON(%s))
        """, ("This is not JSON",))  # ❌ Invalid JSON
    except Exception as e:
        print(f"   Error: {type(e).__name__}")
        print("   ✅ Solution: Always validate JSON before inserting\n")
    
    # ✅ Correct approach
    print("✅ CORRECT: Proper VARIANT insertion")
    try:
        data = {
            'nested': {
                'deeply': {
                    'nested': 'value',
                    'array': [1, 2, 3],
                    'null_value': None,
                    'boolean': True
                }
            }
        }
        
        cursor.execute(f"""
            INSERT INTO {TEST_TABLE} (event_name, user_id, status, event_data)
            VALUES (%s, %s, %s, PARSE_JSON(%s))
        """, (
            'correct_example',
            999,
            'success',
            json.dumps(data)  # ✅ Proper serialization
        ))
        conn.commit()
        print("   Success! Data inserted correctly")
        
        # Verify it's actually VARIANT and can be queried
        cursor.execute(f"""
            SELECT 
                event_data,
                event_data:nested.deeply.nested::STRING as nested_value,
                event_data:nested.deeply.array[0]::INTEGER as first_array_item
            FROM {TEST_TABLE}
            WHERE event_name = 'correct_example'
        """)
        result = cursor.fetchone()
        print(f"   Full VARIANT: {result[0]}")
        print(f"   Nested Value: {result[1]}")
        print(f"   Array Item: {result[2]}")
        
    except Exception as e:
        print(f"   Error: {e}")
    finally:
        cursor.close()
        conn.close()

test_common_errors()

## 8. Best Practices Summary

### ✅ DO:
1. **Always use `json.dumps()`** to serialize Python objects
2. **Use `PARSE_JSON()`** in your SQL INSERT statements
3. **Validate JSON** before inserting (use `json.loads()` to test)
4. **Handle NULL values** explicitly
5. **Use raw connector** for complex VARIANT operations

### ❌ DON'T:
1. **Don't insert dict directly** without serialization
2. **Don't forget `PARSE_JSON()`** in SQL
3. **Don't insert invalid JSON strings**
4. **Don't assume ORMs handle VARIANT automatically**

### 📝 Code Template:

```python
# 1. Prepare your data
variant_data = {
    'key1': 'value1',
    'nested': {'key2': 'value2'}
}

# 2. Serialize to JSON string
json_string = json.dumps(variant_data)

# 3. Use PARSE_JSON in SQL
cursor.execute(
    "INSERT INTO table (col) VALUES (PARSE_JSON(%s))",
    (json_string,)
)
```

## 9. Query Examples

How to query VARIANT data after insertion.

In [ ]:
def query_variant_examples():
    """Examples of querying VARIANT columns"""
    conn = snowflake.connector.connect(**SNOWFLAKE_CONFIG)
    cursor = conn.cursor(DictCursor)
    
    try:
        print("📊 VARIANT Query Examples:\n")
        
        # Example 1: Extract nested value
        print("1️⃣ Extract nested value:")
        cursor.execute(f"""
            SELECT 
                event_name,
                event_data:browser::STRING as browser,
                user_metadata:preferences.theme::STRING as theme
            FROM {TEST_TABLE}
            WHERE event_data:browser IS NOT NULL
            LIMIT 1
        """)
        print(cursor.fetchone())
        
        # Example 2: Extract array element
        print("\n2️⃣ Extract array element:")
        cursor.execute(f"""
            SELECT 
                event_name,
                user_metadata:tags[0]::STRING as first_tag,
                user_metadata:tags[1]::STRING as second_tag
            FROM {TEST_TABLE}
            WHERE user_metadata:tags IS NOT NULL
            LIMIT 1
        """)
        print(cursor.fetchone())
        
        # Example 3: Filter by VARIANT field
        print("\n3️⃣ Filter by VARIANT field:")
        cursor.execute(f"""
            SELECT 
                event_name,
                event_data:page::STRING as page
            FROM {TEST_TABLE}
            WHERE event_data:page = '/dashboard'
        """)
        results = cursor.fetchall()
        for row in results:
            print(row)
        
        # Example 4: Check if key exists
        print("\n4️⃣ Check if key exists:")
        cursor.execute(f"""
            SELECT 
                event_name,
                CASE 
                    WHEN event_data:browser IS NOT NULL THEN 'Has browser'
                    ELSE 'No browser'
                END as browser_check
            FROM {TEST_TABLE}
            LIMIT 3
        """)
        results = cursor.fetchall()
        for row in results:
            print(row)
        
        # Example 5: Flatten array
        print("\n5️⃣ Flatten ARRAY in VARIANT:")
        cursor.execute(f"""
            SELECT 
                event_name,
                f.value::STRING as tag
            FROM {TEST_TABLE},
            LATERAL FLATTEN(input => user_metadata:tags) f
            WHERE user_metadata:tags IS NOT NULL
        """)
        results = cursor.fetchall()
        for row in results:
            print(row)
            
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
    finally:
        cursor.close()
        conn.close()

query_variant_examples()

## 10. Cleanup

In [ ]:
def view_all_data():
    """View all inserted data"""
    conn = snowflake.connector.connect(**SNOWFLAKE_CONFIG)
    cursor = conn.cursor()
    
    try:
        cursor.execute(f"SELECT COUNT(*) FROM {TEST_TABLE}")
        count = cursor.fetchone()[0]
        print(f"📊 Total rows in {TEST_TABLE}: {count}")
        
        cursor.execute(f"""
            SELECT 
                id,
                event_name,
                user_id,
                status,
                event_data
            FROM {TEST_TABLE}
            ORDER BY id
        """)
        
        results = cursor.fetchall()
        print("\n📋 All Data:")
        for row in results:
            print(f"ID: {row[0]}, Event: {row[1]}, User: {row[2]}, Status: {row[3]}")
            
    finally:
        cursor.close()
        conn.close()

view_all_data()

In [ ]:
def drop_test_table():
    """Drop the test table (cleanup)"""
    conn = snowflake.connector.connect(**SNOWFLAKE_CONFIG)
    cursor = conn.cursor()
    
    try:
        cursor.execute(f"DROP TABLE IF EXISTS {TEST_TABLE}")
        print(f"✅ Dropped table {TEST_TABLE}")
    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        cursor.close()
        conn.close()

# Uncomment to drop the test table
# drop_test_table()